In [1]:
!pip install pandas scikit-learn nltk numpy joblib re time

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import time
from joblib import Parallel, delayed

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [17]:
file_path = '/content/drive/MyDrive/Colab Notebooks/News_Category_Dataset_v3.json'
df = pd.read_json(file_path, lines=True)
df['text'] = df['headline'] + " " + df['short_description']
df = df[['category', 'text']]

In [22]:
print("Preprocessing the text...")
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

df['processed_text'] = df['text'].apply(preprocess_text)
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['category'])

Preprocessing the text...


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'], df['category_encoded'], test_size=0.2, random_state=45
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 3), max_features=100000, min_df=3, max_df=0.9, sublinear_tf=True)),
    ('classifier', SGDClassifier(loss='modified_huber', penalty='elasticnet', alpha=1e-5, max_iter=1000, tol=1e-3, random_state=45))
])

In [ ]:
print("Training the model...")
start_time = time.time()
pipeline.fit(X_train, y_train)
end_time = time.time()
training_time = end_time - start_time
print(f"Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

print("Making predictions...")
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

print(f"Total number of samples: {len(df)}")
print(f"Number of training samples: {len(X_train)}")
print(f"Number of testing samples: {len(X_test)}")